# Feature Engineering

Create customer-level features from `merged_transactions.csv` and save `customer_features_prepared.csv`.

This notebook was extracted from `Combined.ipynb` and organized as a standalone stage.

## Notebook Guide
This notebook aggregates cleaned transactions into customer-level features that summarize value, frequency, promotion behavior, and store loyalty.


In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'dataset'
GENERATED_DIR = PROJECT_ROOT / 'generated'
FIGURES_DIR = PROJECT_ROOT / 'figures'

GENERATED_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset dir: {DATA_DIR}')
print(f'Generated dir: {GENERATED_DIR}')
print(f'Figures dir: {FIGURES_DIR}')

import pandas as pd
import numpy as np

df_transactions = pd.read_csv(GENERATED_DIR / 'merged_transactions.csv')
print(f"Loaded merged transactions: {df_transactions.shape}")


Project root: /Users/ponnimuthukumarasamy/Downloads/Prosol Dataset Complete/Sales-Data-Analysis
Dataset dir: /Users/ponnimuthukumarasamy/Downloads/Prosol Dataset Complete/Sales-Data-Analysis/dataset
Generated dir: /Users/ponnimuthukumarasamy/Downloads/Prosol Dataset Complete/Sales-Data-Analysis/generated
Figures dir: /Users/ponnimuthukumarasamy/Downloads/Prosol Dataset Complete/Sales-Data-Analysis/figures
Loaded merged transactions: (1304346, 35)


In [2]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [3]:
# !pip install --upgrade numexpr

## PART 3: FEATURE ENGINEERING
Set up the feature-building workflow that converts transactions into customer summaries.


In [4]:
# ============================================================================
# PART 3: FEATURE ENGINEERING
# ============================================================================

print("\n" + "=" * 80)
print("PART 3: FEATURE ENGINEERING")
print("=" * 80)


PART 3: FEATURE ENGINEERING


In [5]:
df_transactions.columns

Index(['NO_TOKEN_CB', 'CD_TICKET_UNIQUE', 'DT_VENTE', 'TS_VENTE',
       'ID_MAG_TIERS', 'ID_ARTICLE', 'MT_TTC_NET', 'TX_TVA', 'QT_UVC',
       'QT_POIDS', 'IS_PROMO_NATIONALE', 'IS_PROMO_MAGASIN', 'HOUR',
       'DAY_OF_WEEK', 'DAY_NAME', 'MONTH', 'YEAR', 'IS_WEEKEND', 'LB_METIER',
       'LB_SOUS_RAYON', 'LB_FAMILLE', 'LB_SOUS_FAMILLE', 'CD_ARTICLE',
       'LB_ARTICLE', 'ID_LIEU_DE_VENTE', 'LB_DESIGNATION_LIEU_DE_VENTE',
       'LB_ADRESSE', 'CD_POSTAL', 'LB_VILLE', 'LATITUDE', 'LONGITUDE',
       'DT_OUVERTURE', 'TOTAL_QTY', 'QTY_PROMO_NAT', 'QTY_PROMO_STORE'],
      dtype='str')

## STEP 3.1: Customer-Level Feature Engineering
Aggregate transactional behavior into core customer metrics.


In [6]:
# ============================================================================
# STEP 3.1: Customer-Level Feature Engineering
# ============================================================================

print("\n[STEP 3.1] Engineering customer-level features...")
df_transactions['DT_VENTE'] = pd.to_datetime(df_transactions['DT_VENTE'])
df_transactions['TS_VENTE'] = pd.to_datetime(df_transactions['TS_VENTE'], errors='coerce')


# Set Reference Date (Snapshot Date)
reference_date = df_transactions['DT_VENTE'].max() + pd.Timedelta(days=1)
print(f"Reference date for recency: {reference_date}")

# Define Aggregation Dictionary
agg_dict = {
    # === RECENCY ===
    'DT_VENTE': [
        ('first_purchase', 'min'),
        ('last_purchase', 'max'),
        ('shopping_days', 'nunique')
    ],

    # === FREQUENCY ===
    'CD_TICKET_UNIQUE': [
        ('total_transactions', 'nunique')
    ],

    # === MONETARY ===
    'MT_TTC_NET': [
        ('total_spend', 'sum'),
        ('avg_transaction_value', 'mean'),
    ],

    # === PRODUCT DIVERSITY ===
    'LB_METIER': [('dept_diversity', 'nunique')],
    'LB_FAMILLE': [('family_diversity', 'nunique')],

    # === QUANTITY PATTERNS ===
    'TOTAL_QTY': [('total_items_bought', 'sum')], # [NEW] Based on Meeting
    'QT_UVC': [('total_units', 'sum')],
    'QT_POIDS': [('total_weight', 'sum')],

    # === STORE BEHAVIOR ===
    'ID_MAG_TIERS': [('stores_visited', 'nunique')], # Using Technical ID

    # === PROMOTIONAL BEHAVIOR (Quantity Based) ===
    'QTY_PROMO_NAT': [('nat_promo_qty', 'sum')],
    'QTY_PROMO_STORE': [('store_promo_qty', 'sum')]
}

# Perform Groupby
customer_features = df_transactions.groupby('NO_TOKEN_CB').agg(agg_dict).reset_index()

# Flatten column names
customer_features.columns = ['_'.join(col).strip('_') for col in customer_features.columns.values]
customer_features.rename(columns={'NO_TOKEN_CB': 'NO_TOKEN'}, inplace=True)

print(f"✓ Base features created for {len(customer_features)} customers")


[STEP 3.1] Engineering customer-level features...
Reference date for recency: 2025-12-24 00:00:00
✓ Base features created for 121112 customers


In [7]:
customer_features.head(4)

,NO_TOKEN,DT_VENTE_first_purchase,DT_VENTE_last_purchase,DT_VENTE_shopping_days,CD_TICKET_UNIQUE_total_transactions,MT_TTC_NET_total_spend,MT_TTC_NET_avg_transaction_value,LB_METIER_dept_diversity,LB_FAMILLE_family_diversity,TOTAL_QTY_total_items_bought,QT_UVC_total_units,QT_POIDS_total_weight,ID_MAG_TIERS_stores_visited,QTY_PROMO_NAT_nat_promo_qty,QTY_PROMO_STORE_store_promo_qty
0,00000ED6FFB345A27CF94114C9819546E9E65D333D0D10...,2025-12-09,2025-12-10,2,2,26.59,1.564118,3,10,19.003,15.61,3.393,3,3.445,0.0
1,00000F282C0A8BE112B1CB5EBCEB61673125F374902663...,2025-12-05,2025-12-23,4,4,71.55,3.110870,4,17,32.997,21.98,11.017,4,0.000,0.0
2,00005F300675C1E7BA7847B7759248F908230E87BC1EE3...,2025-12-21,2025-12-21,1,1,37.04,6.173333,3,5,8.460,7.73,0.730,3,0.000,0.0
3,0001421783EEDB9B602293092CB337D3A69B84C3B42712...,2025-12-18,2025-12-22,2,2,49.39,4.490000,4,9,12.586,11.56,1.026,4,0.000,0.0


## STEP 3.2: Derived Features
Create higher-level indicators that are more informative for clustering and interpretation.


In [8]:
# ============================================================================
# STEP 3.2: Derived Features
# ============================================================================

print("\n[STEP 3.2] Creating derived features...")

# 1. Recency (days since last purchase)
customer_features['recency_days'] = (
    reference_date - customer_features['DT_VENTE_last_purchase']
).dt.days

# 2. Customer Lifetime
customer_features['customer_lifetime_days'] = (
    customer_features['DT_VENTE_last_purchase'] -
    customer_features['DT_VENTE_first_purchase']
).dt.days

# 3. Frequency (Transactions per month)
customer_features['transactions_per_month'] = (
    customer_features['CD_TICKET_UNIQUE_total_transactions'] /
    ((customer_features['customer_lifetime_days'] / 30) + 1)
)

# 4. Average Basket Size (Items per basket) -> Better than spend for operational view
customer_features['avg_items_per_basket'] = (
    customer_features['TOTAL_QTY_total_items_bought'] /
    customer_features['CD_TICKET_UNIQUE_total_transactions']
)

# 5. Average Spend per Basket
customer_features['avg_basket_value'] = (
    customer_features['MT_TTC_NET_total_spend'] /
    customer_features['CD_TICKET_UNIQUE_total_transactions']
)

# 6. Unit vs Weight Ratio
customer_features['unit_weight_ratio'] = (
    customer_features['QT_UVC_total_units'] /
    (customer_features['QT_POIDS_total_weight'] + 1)
)

# 7. Promotional Behavior
# Formula: (Quantity of Promo Items / Total Quantity) * 100
customer_features['promo_percentage'] = (
    (customer_features['QTY_PROMO_NAT_nat_promo_qty'] +
     customer_features['QTY_PROMO_STORE_store_promo_qty']) /
    (customer_features['TOTAL_QTY_total_items_bought'] + 1e-9) * 100
).clip(upper=100)

# 8. National vs Store Promo Ratio
customer_features['national_vs_store_promo_ratio'] = (
    customer_features['QTY_PROMO_NAT_nat_promo_qty'] /
    (customer_features['QTY_PROMO_STORE_store_promo_qty'] + 1)
)

# 9. Store Loyalty Score (Fixed KeyError)
# Using ID_MAG_TIERS_stores_visited as defined in agg_dict
customer_features['store_loyalty_score'] = 1 / customer_features['ID_MAG_TIERS_stores_visited']

# 10. Product Diversity Score
customer_features['product_diversity_score'] = (
    customer_features['LB_METIER_dept_diversity'] * customer_features['LB_FAMILLE_family_diversity']
)

# 11. Shopping Regularity (Inter-visit time analysis suggested in meeting)
customer_features['avg_days_between_visits'] = (
    customer_features['customer_lifetime_days'] /
    (customer_features['CD_TICKET_UNIQUE_total_transactions'].replace(1, np.nan)) # Avoid div by zero for single-visit customers
).fillna(0)


print("✓ Derived features created.")
print(customer_features[['promo_percentage', 'store_loyalty_score', 'avg_days_between_visits']].describe())


[STEP 3.2] Creating derived features...
✓ Derived features created.
       promo_percentage  store_loyalty_score  avg_days_between_visits
count     121112.000000        121112.000000            121112.000000
mean           9.331353             0.577109                 1.066446
std           22.035910             0.287271                 2.087804
min            0.000000             0.090909                 0.000000
25%            0.000000             0.333333                 0.000000
50%            0.000000             0.500000                 0.000000
75%            6.677022             1.000000                 1.000000
max          100.000000             1.000000                11.000000


## STEP 3.3: Handle Missing Values and Outliers
Stabilize the feature set so extreme values and nulls do not dominate the segmentation.


In [9]:
# ============================================================================
# STEP 3.3: Handle Missing Values and Outliers
# ============================================================================

print("\n[STEP 3.3] Handling missing values and outliers...")

# Fill NaNs created by division
customer_features.fillna(0, inplace=True)
customer_features.replace([np.inf, -np.inf], 0, inplace=True)

# Outlier Detection (not drop high value customers)
outlier_features = ['MT_TTC_NET_total_spend', 'recency_days', 'promo_percentage']

print("\nOutlier Check (Observation):")
for feature in outlier_features:
    Q1 = customer_features[feature].quantile(0.25)
    Q3 = customer_features[feature].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((customer_features[feature] < (Q1 - 1.5 * IQR)) | (customer_features[feature] > (Q3 + 1.5 * IQR))).sum()
    print(f"  {feature}: {outliers} potential outliers")


[STEP 3.3] Handling missing values and outliers...

Outlier Check (Observation):
  MT_TTC_NET_total_spend: 7518 potential outliers
  recency_days: 0 potential outliers
  promo_percentage: 18951 potential outliers


In [10]:
customer_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 121112 entries, 0 to 121111
Data columns (total 26 columns):
 #   Column                               Non-Null Count   Dtype         
---  ------                               --------------   -----         
 0   NO_TOKEN                             121112 non-null  str           
 1   DT_VENTE_first_purchase              121112 non-null  datetime64[us]
 2   DT_VENTE_last_purchase               121112 non-null  datetime64[us]
 3   DT_VENTE_shopping_days               121112 non-null  int64         
 4   CD_TICKET_UNIQUE_total_transactions  121112 non-null  int64         
 5   MT_TTC_NET_total_spend               121112 non-null  float64       
 6   MT_TTC_NET_avg_transaction_value     121112 non-null  float64       
 7   LB_METIER_dept_diversity             121112 non-null  int64         
 8   LB_FAMILLE_family_diversity          121112 non-null  int64         
 9   TOTAL_QTY_total_items_bought         121112 non-null  float64       
 10  QT_UVC_

In [11]:
output_path = GENERATED_DIR / 'customer_features_prepared.csv'
customer_features.to_csv(output_path, index=False)
print(f"✓ Saved customer features to '{output_path}'")


✓ Saved customer features to '/Users/ponnimuthukumarasamy/Downloads/Prosol Dataset Complete/Sales-Data-Analysis/generated/customer_features_prepared.csv'


## Results Summary

- Aggregated the cleaned transactions into customer-level features covering spend, frequency, basket behavior, promotion usage, and store loyalty.
- Derived higher-level metrics such as promo share, loyalty concentration, and average time between visits.
- Reviewed missing values and outlier behavior before saving the prepared feature table used for clustering.
- Main output: `generated/customer_features_prepared.csv`.
